# 01 (keshav) — current 01 logic + IndicXlit hybrid transliteration

Identical to `01_eda_preprocessing.ipynb` (diverse sampling included) except for one change: `normalize_name` / `normalize_address` transliterate native-script text with **IndicXlit** (neural, word-cached) instead of `anyascii` alone, when a checkpoint is available. This is keshav_preprocessing.ipynb's hybrid engine, spliced into our current 01 unchanged.

* `XLIT_DIR` unset and no checkpoint under `/kaggle/input` → `XLIT_MODEL` stays `None` → byte-identical to plain 01 (anyascii-only).
* `XLIT_AUTO_DOWNLOAD=true` (default) downloads the ~124MB IndicXlit indic→en checkpoint once.
* Measured on the full competition dataset: only ~1,550 unique native-script word tokens exist in total (synthetic/templated data), so the added decode cost is a few minutes, one time, regardless of sample size — see `docs/` for the write-up. Output quality is mixed (some words come out clean, some garbled); this notebook is for testing that trade-off, not a recommended default.

Everything else — sampling, tags, output contract — is unchanged from `01_eda_preprocessing.ipynb`.

# 01 — EDA + Preprocessing (full preprocessing.md coverage, Kaggle-ready)

Business Entity Resolution: `S1 -> S2/S3`, macro-F0.5. Run top-to-bottom locally or on Kaggle.

- **Local**: data auto-found at `../6ab10eb3b23ba_student_resource/student_resource/dataset` (or `DATA_ROOT` env).
- **Kaggle**: upload TSVs as a dataset, set `DATA_ROOT=/kaggle/input/<dataset>`.
- `FULL_RUN=true` processes every row chunked into **per-country parquet** (`preprocessing.md` §4 contract). Default (`false`) runs the full EDA and processes a **sample**.
- **Sample size and kind** (used when `FULL_RUN=false`; first code cell or environment variables):

  | Setting | Default | Meaning |
  |---|---|---|
  | `SAMPLE_SIZE` | `40k` | **S1** rows per country: a number (`250000`) or `20k` / `100k` / `1.5m`. Each S2/S3 file gets the same ratio to S1 as the full data (about 2.3x S1 for each of S2 and S3), so the sample keeps the real share of records per S1 |
  | `SAMPLE_MODE` | `linked` | `linked`: random train S1, then each train S2/S3 file gets **all true matches of those S1** first, then random other records up to its size, so the sample has real pairs and real distractors (test files: random rows), so the sample has real pairs to learn from; `random`: random rows; `head`: the first rows (fastest, almost no true pairs) |
  | `SAMPLE_DIVERSE` | `true` | stratify every random draw above (S1 selection, the "other records" distractor fill, and trimming an over-quota true-match set) across native-script name, domain-as-name, honorific-prefix and, for train S1, match-count (singleton / few / many). Rare cases still get a small guaranteed share instead of being left to chance at small sample sizes. `false` = plain uniform random, the old behavior |
  | `PROCESSED_OUT` | `processed` | output folder (e.g. `processed_100k`, to keep a full `processed/` untouched) |
  | `PROCESS_SPLITS` | `train,test` | `train` = preprocess only the train files (experiments; a submission needs test too) |

  The competition files are found by name under `DATA_ROOT`, in one flat folder or in `train/` and `test/` subfolders.

  The EDA cells always read the full files; only the processing step uses the sample.
- Covers preprocessing.md §2 (EDA checklist), §3 (normalization), §3.3 (native-script), §5 (done definition).

In [ ]:
import os, re, json, unicodedata
from pathlib import Path

import polars as pl

REPO = Path.cwd()
_env_root = os.environ.get("DATA_ROOT", "").strip()
# the competition TSVs, found by name under DATA_ROOT (or the usual local / Kaggle folders), in a flat folder
# or in train/ and test/ subfolders
_ROOTS = ([Path(_env_root)] if _env_root else []) + [
    REPO / "../6ab10eb3b23ba_student_resource/student_resource/dataset",
    REPO.parent / "6ab10eb3b23ba_student_resource/student_resource/dataset",
    REPO.parent / "Data/student_resource/dataset", Path("/kaggle/input")]
FILE_NAMES = {"s1tr": "train_source1.tsv", "s2tr": "train_source2.tsv", "s3tr": "train_source3.tsv",
              "gt": "train_ground_truth.tsv", "s1te": "test_source1.tsv", "s2te": "test_source2.tsv",
              "s3te": "test_source3.tsv"}


def find_data_file(name):
    for root in _ROOTS:
        if root.exists():
            hits = sorted((h for h in root.glob(f"**/{name}") if "__MACOSX" not in str(h)), key=lambda h: len(str(h)))
            if hits:
                return hits[0].resolve()
    return None


P = {k: find_data_file(v) for k, v in FILE_NAMES.items()}
assert all(P.values()), f"competition files not found (set DATA_ROOT): {[FILE_NAMES[k] for k, v in P.items() if v is None]}"
DATA_ROOT = P["s1tr"].parent
FULL_RUN = os.environ.get("FULL_RUN", "false").lower() == "true"
OUT = Path(os.environ.get("PROCESSED_OUT") or REPO / "processed")
# ---- Sample (used when FULL_RUN is false) ----------------------------------------------------------------
SAMPLE_SIZE = os.environ.get("SAMPLE_SIZE", "40k")        # S1 rows per country (S2/S3 keep the full data's ratio to S1)
SAMPLE_MODE = os.environ.get("SAMPLE_MODE", "linked")     # linked | random | head
SAMPLE_DIVERSE = os.environ.get("SAMPLE_DIVERSE", "true").lower() == "true"   # stratify random draws, see cell 0
SEED = 42
# ---------------------------------------------------------------------------------------------------------
assert SAMPLE_MODE in {"linked", "random", "head"}, SAMPLE_MODE


def parse_size(value):
    """'40k' -> 40000, '1.5m' -> 1500000, '250000' -> 250000."""
    v = str(value).strip().lower().replace("_", "").replace(",", "")
    scale = {"k": 1_000, "m": 1_000_000}.get(v[-1:], 1)
    return int(float(v[:-1] if scale > 1 else v) * scale)


SAMPLE_N_PER_COUNTRY = parse_size(SAMPLE_SIZE)
OUT.mkdir(parents=True, exist_ok=True)
SEP, SAMPLE_N, CHUNK = "\t", 100000, 200000
PROCESS_SPLITS = [x.strip() for x in os.environ.get("PROCESS_SPLITS", "train,test").split(",") if x.strip()]   # train,test | train (experiments)
print("REPO=", str(REPO))
print("DATA_ROOT=", str(DATA_ROOT))
print("FULL_RUN=", FULL_RUN, "| output:", OUT)
if not FULL_RUN:
    print(f"sample: {SAMPLE_N_PER_COUNTRY:,} S1 per country (S2/S3 in the full data's ratio to S1), mode {SAMPLE_MODE} (diverse={SAMPLE_DIVERSE})")
print("missing=", [k for k, v in P.items() if not Path(v).exists()] or "none")

def safe(x, n=120):
    s = repr(x)
    return s[:n]


# ---- Diversity: tag each row with a small stratum (script / name shape / match count) so a random
# draw can guarantee rare cases a minimum share instead of leaving them to chance. Runs on a frame
# already fully collected in memory (every caller below collects before sampling), so this adds no
# extra I/O -- just cheap vectorized string ops over what's already there.
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
_LATIN_CLASS = "".join(chr(a) + "-" + chr(b) for a, b in _LATIN_RANGES)
NATIVE_RE_PL = f"[^{_LATIN_CLASS}]"                      # polars/rust regex: any char outside name-safe Latin


def add_case_tag(df):
    """`_case`: native-script name / domain-as-name / honorific-prefix / plain (priority in that order)."""
    name = pl.col("business_name").fill_null("")
    return df.with_columns(
        pl.when(name.str.contains(NATIVE_RE_PL)).then(pl.lit("native"))
        .when(name.str.to_lowercase().str.contains(r"(?i)\.(com|in|fr|net|org)\b")).then(pl.lit("domain"))
        .when(name.str.contains(r"(?i)^\s*(smt|shri|mr|mrs|miss|dr)\.?\s")).then(pl.lit("honorific"))
        .otherwise(pl.lit("plain")).alias("_case"))


def add_mc_tag(df, mc_by_s1):
    """`_mc`: train-S1 match-count bucket (0 / 1-2 / 3-5 / 6+), from the exploded ground-truth `pairs`."""
    return df.join(mc_by_s1, on="entity_id", how="left").with_columns(
        pl.when(pl.col("_mc").is_null() | (pl.col("_mc") == 0)).then(pl.lit("mc0"))
        .when(pl.col("_mc") <= 2).then(pl.lit("mc1-2"))
        .when(pl.col("_mc") <= 5).then(pl.lit("mc3-5"))
        .otherwise(pl.lit("mc6+")).alias("_mc"))


def stratified_sample(df, n, tag_cols, seed):
    """Sample n rows of df, giving every stratum in tag_cols (columns already on df, from add_case_tag /
    add_mc_tag) a small guaranteed floor before filling the rest proportionally to each stratum's real
    share -- so a rare case (native-script names, singleton S1, ...) still shows up at small sample sizes
    instead of vanishing by chance. Plain random when SAMPLE_DIVERSE is off or n covers the whole frame;
    tag_cols are dropped from the result either way, so callers get back the original columns."""
    if not SAMPLE_DIVERSE or n >= len(df):
        return df.sample(min(n, len(df)), seed=seed).drop(tag_cols)
    df = df.with_columns(pl.concat_str(tag_cols, separator="|").alias("_stratum"))
    sizes = {r["_stratum"]: r["len"] for r in df.group_by("_stratum").len().to_dicts()}
    floor = max(20, round(n * 0.01))
    chosen = {s: df.filter(pl.col("_stratum") == s).sample(min(sz, floor), seed=seed) for s, sz in sizes.items()}
    left = n - sum(len(v) for v in chosen.values())
    pool = {s: sizes[s] - len(chosen[s]) for s in sizes}
    total_pool = sum(pool.values())
    if left > 0 and total_pool > 0:
        for s, pool_n in pool.items():
            extra = min(pool_n, round(left * pool_n / total_pool)) if pool_n > 0 else 0
            if extra <= 0:
                continue
            more = df.filter(pl.col("_stratum") == s).join(chosen[s].select("entity_id"), on="entity_id", how="anti")
            chosen[s] = pl.concat([chosen[s], more.sample(min(extra, len(more)), seed=seed)])
    out = pl.concat(list(chosen.values()), how="diagonal").unique(subset=["entity_id"])
    if len(out) > n:
        out = out.sample(n, seed=seed)
    elif len(out) < n:                                    # rounding shortfall: top up randomly from what's left
        rest = df.join(out.select("entity_id"), on="entity_id", how="anti")
        out = pl.concat([out, rest.sample(min(n - len(out), len(rest)), seed=seed)], how="diagonal")
    return out.drop(tag_cols + ["_stratum"])


In [ ]:
# ---- EDA basics: shapes, schema, prefixes, nulls, ground truth ----
eda = {}
FILES = ["s1tr", "s2tr", "s3tr", "s1te", "s2te", "s3te"]
for k in FILES:
    n = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(pl.len()).collect().item()
    eda[k + "_rows"] = n
    print(k, "rows=", n)
sch = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000).collect_schema()
eda["columns"] = sch.names()
print("columns=", eda["columns"])
PX = {"s1tr": "S1-", "s2tr": "S2-", "s3tr": "S3-", "s1te": "S1-", "s2te": "S2-", "s3te": "S3-"}
eda["prefix_bad"] = {}
for k, px in PX.items():
    bad = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).filter(
        ~pl.col("entity_id").str.starts_with(px)).select(pl.len()).collect().item()
    eda["prefix_bad"][k] = bad
print("prefix_mismatches=", eda["prefix_bad"])
for k in FILES:
    d = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(
        [pl.col(c).is_null().sum().alias(c) for c in ["business_name", "business_address"]]).collect().to_dicts()[0]
    eda[k + "_nulls"] = d
    print(k, "nulls=", d)
gt = pl.read_csv(str(P["gt"]), separator="\t", infer_schema_length=10000).with_columns(
    pl.col("matched_entity_ids").fill_null("").str.strip_chars().alias("m"))
gt = gt.with_columns(
    pl.when(pl.col("m") == "").then(0).otherwise(pl.col("m").str.split(",").list.len()).alias("mc"))
mc = gt["mc"]
eda["gt_mean_mc"] = round(float(mc.mean()), 3)
eda["gt_max_mc"] = int(mc.max())
eda["gt_singleton_pct"] = round(float((mc == 0).mean() * 100), 2)
eda["gt_dist"] = {int(r["mc"]): int(r["len"]) for r in gt.group_by("mc").len().sort("mc").head(12).to_dicts()}
print("mean_mc=", eda["gt_mean_mc"], "max=", eda["gt_max_mc"], "singleton%=", eda["gt_singleton_pct"])
json.dump(eda, open(OUT / "eda_summary.json", "w"), indent=1)
print("basics done")


In [ ]:
# ---- EDA structural: S2/S3 share, reuse=1, distractors, cross-country, lengths, FR samples ----
# S2 vs S3 share without exploding (string counting over full GT)
agg = gt.select([pl.col("m").str.count_matches("S2-").sum().alias("s2pairs"),
                 pl.col("m").str.count_matches("S3-").sum().alias("s3pairs")]).to_dicts()[0]
eda["s2_pairs"], eda["s3_pairs"] = int(agg["s2pairs"]), int(agg["s3pairs"])
tot = eda["s2_pairs"] + eda["s3_pairs"]
eda["s2_share_pct"] = round(100 * eda["s2_pairs"] / tot, 1)
eda["s3_share_pct"] = round(100 * eda["s3_pairs"] / tot, 1)
print("pairs S2=", eda["s2_pairs"], "S3=", eda["s3_pairs"], "share=", eda["s2_share_pct"], "/", eda["s3_share_pct"])
# Explode once: reuse check + distractor math
pairs = gt.select(["source1_entity_id", "m"]).filter(pl.col("m") != "").select(
    [pl.col("source1_entity_id").alias("s1"),
     pl.col("m").str.split(",").alias("lst")]).explode("lst").rename({"lst": "mid"})
vc = pairs.group_by("mid").len()
eda["reuse_max"] = int(vc["len"].max())
eda["matched_unique"] = int(vc.height)
s2s3_total = eda["s2tr_rows"] + eda["s3tr_rows"]
eda["distractor_pct"] = round(100 * (s2s3_total - eda["matched_unique"]) / s2s3_total, 2)
print("reuse_max=", eda["reuse_max"], "matched_unique=", eda["matched_unique"],
      "distractor%=", eda["distractor_pct"])
# Cross-country check: join pair countries (S1 map 2.2M + S2/S3 maps, id+country only)
s1map = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000).select(["entity_id", "country"]).collect()
s2map = pl.scan_csv(str(P["s2tr"]), separator="\t", infer_schema_length=10000).select(["entity_id", "country"]).collect()
s3map = pl.scan_csv(str(P["s3tr"]), separator="\t", infer_schema_length=10000).select(["entity_id", "country"]).collect()
p2 = pairs.filter(pl.col("mid").str.starts_with("S2-")).join(s1map, left_on="s1", right_on="entity_id", how="left").rename({"country": "c1"})
p2 = p2.join(s2map, left_on="mid", right_on="entity_id", how="left").rename({"country": "c2"})
p3 = pairs.filter(pl.col("mid").str.starts_with("S3-")).join(s1map, left_on="s1", right_on="entity_id", how="left").rename({"country": "c1"})
p3 = p3.join(s3map, left_on="mid", right_on="entity_id", how="left").rename({"country": "c2"})
eda["cross_country_mismatches"] = int(p2.filter(pl.col("c1") != pl.col("c2")).height + p3.filter(pl.col("c1") != pl.col("c2")).height)
eda["cross_country_checked"] = int(p2.height + p3.height)
print("cross-country mismatches=", eda["cross_country_mismatches"], "checked=", eda["cross_country_checked"])
del p2, p3
# Lengths per source (20k samples: char + token medians)
for k in ["s1tr", "s2tr", "s3tr"]:
    s = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(
        ["business_name", "business_address"]).head(20000).collect().with_columns(
        [pl.col("business_name").fill_null("").str.len_chars().alias("nl"),
         pl.col("business_name").fill_null("").str.split(" ").list.len().alias("nt"),
         pl.col("business_address").fill_null("").str.len_chars().alias("al")])
    eda[k + "_len"] = {"name_char_med": float(s["nl"].median()), "name_tok_med": float(s["nt"].median()),
                       "addr_char_med": float(s["al"].median())}
    print(k, eda[k + "_len"])
# Completeness per country (30k samples: null-addr rate + PIN-presence rate)
for k in ["s1tr", "s2tr", "s3tr"]:
    s = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).select(
        ["business_address", "country"]).head(30000).collect().with_columns(
        [pl.col("business_address").is_null().alias("null_addr"),
         pl.col("business_address").fill_null("").str.contains(r"\d{5,}").alias("has_pin")])
    eda[k + "_completeness"] = {r["country"]: {"null_addr_pct": round(100 * float(r["null_addr"]), 1),
        "pin_present_pct": round(100 * float(r["has_pin"]), 1)}
        for r in s.group_by("country").agg([pl.col("null_addr").mean(), pl.col("has_pin").mean()]).to_dicts()}
    print(k, eda[k + "_completeness"])
# FR samples: 5 rows per test source
for k in ["s1te", "s2te", "s3te"]:
    print("--- FR sample", k, "---")
    for r in pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).filter(
            pl.col("country") == "France").head(5).collect().to_dicts():
        print("   ", safe(r["business_name"]), "|", safe(r["business_address"]))
# S2/S3 country splits (train + test)
for k in ["s2tr", "s3tr", "s2te", "s3te"]:
    d = pl.scan_csv(str(P[k]), separator="\t", infer_schema_length=10000).group_by("country").len().collect().sort("country")
    eda[k + "_country"] = {r["country"]: r["len"] for r in d.to_dicts()}
    print(k, eda[k + "_country"])
# Frequent tokens + legal-suffix counts (100k raw S1 names)
LEGAL_RAW = {"pvt", "private", "ltd", "limited", "corp", "corporation", "inc", "llc", "llp", "co",
             "enterprises", "sarl", "sas", "sa", "eurl", "sci", "groupe"}
tok = pl.scan_csv(str(P["s1tr"]), separator="\t", infer_schema_length=10000).select("business_name").head(
    100000).collect().with_columns(pl.col("business_name").fill_null("").str.to_lowercase().alias("n"))
toks = tok.select(pl.col("n").str.split(" ").alias("t")).explode("t").filter(
    (pl.col("t") != "") & (~pl.col("t").is_in(sorted(LEGAL_RAW))))
eda["top_tokens"] = [[r["t"], int(r["len"])] for r in toks.group_by("t").len().sort("len", descending=True).head(20).to_dicts()]
suf = tok.select(pl.col("n").str.split(" ").list.last().alias("last")).filter(pl.col("last").is_in(sorted(LEGAL_RAW)))
eda["top_legal"] = [[r["last"], int(r["len"])] for r in suf.group_by("last").len().sort("len", descending=True).head(10).to_dicts()]
print("top_tokens=", eda["top_tokens"][:8])
print("top_legal=", eda["top_legal"])
json.dump(eda, open(OUT / "eda_summary.json", "w"), indent=1)
print("structural EDA done")


In [ ]:
# ---- True-pair similarity (20k matched pairs) + 15-pair noise catalog ----
from rapidfuzz import fuzz
pr = pairs.sample(20000, seed=42)
s1ids, mids = pr["s1"].to_list(), pr["mid"].to_list()
s1rec = {r["entity_id"]: r for r in pl.scan_csv(str(P["s1tr"]), separator="\t",
    infer_schema_length=10000).filter(pl.col("entity_id").is_in(s1ids)).collect().to_dicts()}
s2need = [m for m in mids if m.startswith("S2-")]
s3need = [m for m in mids if m.startswith("S3-")]
mx = {}
for kk, pp, need in [("s2", P["s2tr"], s2need), ("s3", P["s3tr"], s3need)]:
    if need:
        for r in pl.scan_csv(str(pp), separator="\t", infer_schema_length=10000).filter(
                pl.col("entity_id").is_in(need)).collect().to_dicts():
            mx[r["entity_id"]] = r
ts, fr, aj = [], [], []
shown = 0
print("--- noise catalog (S1 vs matched S2/S3, 15 pairs) ---")
for a, b in zip(s1ids, mids):
    r1, r2 = s1rec.get(a), mx.get(b)
    if not r1 or not r2:
        continue
    n1, n2 = (r1["business_name"] or "").lower(), (r2["business_name"] or "").lower()
    d1, d2 = (r1["business_address"] or "").lower(), (r2["business_address"] or "").lower()
    ts.append(fuzz.token_set_ratio(n1, n2))
    fr.append(fuzz.ratio(n1, n2))
    t1, t2 = set(d1.replace(",", " ").split()), set(d2.replace(",", " ").split())
    aj.append(len(t1 & t2) / max(1, len(t1 | t2)))
    if shown < 15:
        print(f"[{b}]")
        print("  S1:", safe(r1["business_name"]), "|", safe(r1["business_address"]))
        print("  MT:", safe(r2["business_name"]), "|", safe(r2["business_address"]))
        shown += 1
import statistics as _st
eda["sim_20k"] = {"token_set_mean": round(_st.mean(ts), 1),
    "token_set_gt90_pct": round(100 * sum(1 for x in ts if x >= 90) / len(ts), 1),
    "token_set_lt60_pct": round(100 * sum(1 for x in ts if x < 60) / len(ts), 1),
    "fuzz_mean": round(_st.mean(fr), 1), "addr_jaccard_mean": round(_st.mean(aj), 3), "n": len(ts)}
print("sim_20k=", eda["sim_20k"])
json.dump(eda, open(OUT / "eda_summary.json", "w"), indent=1)
print("similarity EDA done")


In [ ]:
# ---- Hybrid checkpoint bootstrap: download once so Run-All works with no model dataset ----
# Precedence: explicit $XLIT_DIR wins; else the settings cell below auto-discovers under
# /kaggle/input; else download the official v1.0 indic->en bundle here (persists in
# /kaggle/working across saved sessions). Set XLIT_AUTO_DOWNLOAD=false to stay baseline.
import os as _os
from pathlib import Path as _Path
if not _os.environ.get("XLIT_DIR", "").strip() and _os.environ.get("XLIT_AUTO_DOWNLOAD", "true").lower() == "true":
    _dst = _Path.cwd() / "xlit_model"
    if not (_dst / "transformer" / "indicxlit.pt").exists():
        import urllib.request as _urlreq
        import zipfile as _zipmod
        _dst.mkdir(parents=True, exist_ok=True)
        _zp = _dst / "model.zip"
        print("downloading IndicXlit indic->en checkpoint (~124MB)...")
        _urlreq.urlretrieve("https://github.com/AI4Bharat/IndicXlit/releases/download/v1.0/indicxlit-indic-en-v1.0.zip", str(_zp))
        _zipmod.ZipFile(_zp).extractall(_dst)
        _zp.unlink()
        print("extracted to", _dst)
    _os.environ["XLIT_DIR"] = str(_dst)
    print("XLIT_DIR set to", _dst)


In [ ]:
# ---- IndicXlit settings (from keshav_preprocessing.ipynb; unset/not-found == identical to plain notebook) ----
XLIT_DIR = os.environ.get("XLIT_DIR", "").strip()
if not XLIT_DIR:
    import glob as _glob
    _hits = sorted((Path(h) for h in _glob.glob("/kaggle/input/**/corpus-bin", recursive=True)
                    if "__MACOSX" not in h), key=lambda h: len(str(h)))
    if _hits and (_hits[0].parent / "corpus-bin").exists():
        XLIT_DIR = str(_hits[0].parent)
        print("XLIT auto-discovered:", XLIT_DIR)
if not XLIT_DIR:
    print("XLIT disabled (no checkpoint found; attach it and/or set XLIT_DIR to enable hybrid)")
XLIT_LANGS = os.environ.get("XLIT_LANGS", "").strip()


In [ ]:
# ---- Hybrid transliteration engine: IndicXlit (native scripts) + anyascii (Latin/French) ----
# Ported from INDICxLIT-Transliteration.ipynb (validated implementation). Adaptations:
# (1) torch import/install happens ONLY when XLIT_DIR is set, so baseline CPU-only runs
# never pay for torch; (2) XLIT_CACHE keyed by (word, lang); (3) whitespace split.
# XLIT_DIR unset/empty -> XLIT_MODEL stays None -> pipeline == anyascii-only baseline.
import math, os, re, subprocess, sys, unicodedata as _ud
from pathlib import Path

XLIT_MODEL = None
_indic_norms, _NF = {}, None
if XLIT_DIR:
    try:
        import torch
        import torch.nn.functional as F
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"])
        import torch
        import torch.nn.functional as F

    DEVICE = torch.device("cuda" if torch.cuda.is_available()
                           and os.environ.get("XLIT_USE_GPU", "true").lower() == "true" else "cpu")
    print("XLIT_DIR=", XLIT_DIR or "(unset -> hybrid disabled)", "| torch device=", DEVICE,
          "| cuda devices=", torch.cuda.device_count() if torch.cuda.is_available() else 0)
    D, H, HD = 256, 4, 64
    SCALE = math.sqrt(D)          # embed_scale: no_scale_embedding=False in checkpoint
    EPS = 1e-5                    # fairseq LayerNorm default
    PAD, EOS, UNK = 1, 2, 3       # fairseq Dictionary specials: <s>=0,<pad>=1,</s>=2,<unk>=3


    def load_dict(path):
        syms = ["<s>", "<pad>", "</s>", "<unk>"]
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if line:
                    syms.append(line.rsplit(" ", 1)[0])
        return syms


    SORTED_LANGS = sorted("as bn brx en gom gu hi kn ks mai ml mni mr ne or pa sa sd si ta te ur".split())
    BASE_SRC = 780 + 4


    def lang_idx(lang):
        return BASE_SRC + SORTED_LANGS.index(lang)


    def sinusoid(n, device):
        half = D // 2
        freqs = torch.exp(torch.arange(half, device=device).float() * -(math.log(10000) / (half - 1)))
        pos = torch.arange(2, 2 + n, device=device).float().unsqueeze(1)
        e = pos * freqs.unsqueeze(0)
        return torch.cat([torch.sin(e), torch.cos(e)], dim=1)


    def _mha(x, kv, p, mask=None):
        nq, nk = x.shape[0], kv.shape[0]
        Q = (x @ p["q"].T + p["qb"]).view(nq, H, HD).permute(1, 0, 2) * (HD ** -0.5)
        K = (kv @ p["k"].T + p["kb"]).view(nk, H, HD).permute(1, 0, 2)
        V = (kv @ p["v"].T + p["vb"]).view(nk, H, HD).permute(1, 0, 2)
        s = Q @ K.transpose(-1, -2)
        if mask is not None:
            s = s + mask
        o = (torch.softmax(s, dim=-1) @ V).permute(1, 0, 2).reshape(nq, D)
        return o @ p["o"].T + p["ob"]


    def gelu_ffn(x, w1, b1, w2, b2):
        h = x @ w1.T + b1
        return (0.5 * h * (1.0 + torch.erf(h / math.sqrt(2.0)))) @ w2.T + b2


    class XlitModel:
        """Pure-torch inference for IndicXlit indic->en (no fairseq needed)."""

        def __init__(self, ckpt_path, src_dict_path, tgt_dict_path, device=DEVICE):
            raw_sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)["model"]
            self.device = device
            self.sd = {k: v.to(device) for k, v in raw_sd.items()}
            self.src_syms = load_dict(src_dict_path)
            self.tgt_syms = load_dict(tgt_dict_path)
            self.src_id = {s: i for i, s in enumerate(self.src_syms)}
            assert len(self.src_syms) == 784 and len(self.tgt_syms) == 32, (
                f"dict size mismatch: src={len(self.src_syms)} tgt={len(self.tgt_syms)} "
                "-- wrong checkpoint/dicts?")
            self.nl = max(int(k.split(".")[2]) for k in self.sd if k.startswith("encoder.layers.")) + 1

        def _L(self, pre, i, name):
            s = self.sd
            return (lambda x: F.layer_norm(x, (D,), s[f"{pre}.layers.{i}.{name}.weight"],
                                            s[f"{pre}.layers.{i}.{name}.bias"], EPS))

        def _P(self, pre, i, attn):
            s = self.sd
            return {k: s[f"{pre}.layers.{i}.{attn}{suf}"]
                    for k, suf in [("q", ".q_proj.weight"), ("qb", ".q_proj.bias"),
                                   ("k", ".k_proj.weight"), ("kb", ".k_proj.bias"),
                                   ("v", ".v_proj.weight"), ("vb", ".v_proj.bias"),
                                   ("o", ".out_proj.weight"), ("ob", ".out_proj.bias")]}

        @torch.no_grad()
        def encode(self, ids):
            s = self.sd
            x = s["encoder.embed_tokens.weight"][ids] * SCALE + sinusoid(len(ids), self.device)
            x = F.layer_norm(x, (D,), s["encoder.layernorm_embedding.weight"],
                              s["encoder.layernorm_embedding.bias"], EPS)
            for i in range(self.nl):
                x = x + _mha(self._L("encoder", i, "self_attn_layer_norm")(x),
                              self._L("encoder", i, "self_attn_layer_norm")(x),
                              self._P("encoder", i, "self_attn"))
                x = x + gelu_ffn(self._L("encoder", i, "final_layer_norm")(x),
                                  s[f"encoder.layers.{i}.fc1.weight"], s[f"encoder.layers.{i}.fc1.bias"],
                                  s[f"encoder.layers.{i}.fc2.weight"], s[f"encoder.layers.{i}.fc2.bias"])
            return F.layer_norm(x, (D,), s["encoder.layer_norm.weight"],
                                 s["encoder.layer_norm.bias"], EPS)

        @torch.no_grad()
        def _decode_logits(self, ids, enc):
            s = self.sd
            n = len(ids)
            x = s["decoder.embed_tokens.weight"][ids] * SCALE + sinusoid(n, self.device)
            x = F.layer_norm(x, (D,), s["decoder.layernorm_embedding.weight"],
                              s["decoder.layernorm_embedding.bias"], EPS)
            causal = torch.triu(torch.full((n, n), float("-inf"), device=self.device), 1) \
                .unsqueeze(0).expand(H, n, n)
            for i in range(self.nl):
                x = x + _mha(self._L("decoder", i, "self_attn_layer_norm")(x),
                              self._L("decoder", i, "self_attn_layer_norm")(x),
                              self._P("decoder", i, "self_attn"), causal)
                x = x + _mha(self._L("decoder", i, "encoder_attn_layer_norm")(x), enc,
                              self._P("decoder", i, "encoder_attn"))
                x = x + gelu_ffn(self._L("decoder", i, "final_layer_norm")(x),
                                  s[f"decoder.layers.{i}.fc1.weight"], s[f"decoder.layers.{i}.fc1.bias"],
                                  s[f"decoder.layers.{i}.fc2.weight"], s[f"decoder.layers.{i}.fc2.bias"])
            x = F.layer_norm(x, (D,), s["decoder.layer_norm.weight"],
                              s["decoder.layer_norm.bias"], EPS)
            out = x @ s["decoder.output_projection.weight"].T
            if "decoder.output_projection.bias" in s:
                out = out + s["decoder.output_projection.bias"]
            return out

        def encode_word(self, word, lang):
            ids = [lang_idx(lang)] + [self.src_id.get(c, UNK) for c in word] + [lang_idx(lang)]
            return self.encode(torch.tensor(ids, device=self.device))

        @torch.no_grad()
        def greedy(self, word, lang, max_len=60):
            enc = self.encode_word(word, lang)
            out = [EOS]
            for _ in range(max_len):
                nxt = int(self._decode_logits(torch.tensor(out, device=self.device), enc)[-1].argmax())
                if nxt == EOS:
                    break
                out.append(nxt)
            return "".join(self.tgt_syms[i] for i in out[1:])
    def find_xlit_assets(xlit_dir):
        base = Path(xlit_dir) / "corpus-bin"
        ckpt = Path(xlit_dir) / "transformer" / "indicxlit.pt"
        tgt_dict = base / "dict.en.txt"
        src_dict = base / "dict.hi.txt"
        if not src_dict.exists():  # fallback only; all source dicts are byte-identical joint vocabs
            src_dict = next((p for p in sorted(base.glob("dict.*.txt")) if p.name != "dict.en.txt"), None)
        return ckpt, src_dict, tgt_dict



    if XLIT_DIR:
        try:
            from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic-nlp-library"])
            from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
        _NF = IndicNormalizerFactory()
        ckpt, src_dict, tgt_dict = find_xlit_assets(XLIT_DIR)
        if ckpt.exists() and src_dict and tgt_dict.exists():
            XLIT_MODEL = XlitModel(str(ckpt), str(src_dict), str(tgt_dict), device=DEVICE)
            print("XLIT_MODEL loaded from", XLIT_DIR, "| layers=", XLIT_MODEL.nl)
        else:
            print("WARNING: XLIT_DIR set but assets not found under it (expected corpus-bin/dict.*.txt "
                  "+ transformer/indicxlit.pt) -- falling back to anyascii-only baseline.")
else:
    print("XLIT_DIR unset -> hybrid disabled, pipeline == anyascii-only baseline")
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
_CLS = "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES)
RE_NATIVE = re.compile(f"[^{_CLS}]")
RE_LATIN_WORD = re.compile(f"^[{_CLS} ]*$")


def has_native_script(s):
    return bool(RE_NATIVE.search(s or ""))


SCRIPT_LANG = [((0x0900, 0x097F), "hi"), ((0x0C00, 0x0C7F), "te"), ((0x0B80, 0x0BFF), "ta"),
               ((0x0D00, 0x0D7F), "ml"), ((0x0C80, 0x0CFF), "kn"), ((0x0980, 0x09FF), "bn"),
               ((0x0A80, 0x0AFF), "gu"), ((0x0A00, 0x0A7F), "pa"), ((0x0B00, 0x0B7F), "or"),
               ((0x0D80, 0x0DFF), "si"), ((0x0600, 0x06FF), "ur")]


def script_lang(w):
    for (a, b), l in SCRIPT_LANG:
        if any(a <= ord(c) <= b for c in w):
            return l
    return None


LEGAL_ABBR_MAP = {"\u092a\u094d\u0930\u093e": "pvt", "\u0932\u093f": "ltd"}  # प्रा->pvt, लि->ltd



def _indic_normalize(lang, w):
    if lang not in _indic_norms:
        try:
            _indic_norms[lang] = _NF.get_normalizer(lang)
        except Exception:
            _indic_norms[lang] = _NF.get_normalizer("hi")
    return _indic_norms[lang].normalize(w)


def _strip_punct_ws(w):
    # Strip punctuation/separators only -- keep combining marks (vowel signs, virama).
    return "".join(c for c in w if not _ud.category(c).startswith(("P", "Z")))


XLIT_CACHE = {}  # {raw_word: latin} -- cost is bounded by UNIQUE tokens, not rows


def xlit_to_latin(raw_s):
    if not raw_s:
        return raw_s
    out = []
    for w in raw_s.split():
        if not w or RE_LATIN_WORD.match(w):
            out.append(w)
            continue
        plain = _strip_punct_ws(w)
        if plain.lower() in LEGAL_ABBR_MAP:
            out.append(LEGAL_ABBR_MAP[plain.lower()])
            continue
        ck = (w, script_lang(w) or "hi")
        if ck in XLIT_CACHE:
            out.append(XLIT_CACHE[ck])
            continue
        if len(plain) <= 4 or XLIT_MODEL is None or script_lang(w) is None:
            translit = anyascii(w) if HAS_AA else w
        else:
            lang = ck[1]
            wn = _indic_normalize(lang, w)
            translit = XLIT_MODEL.greedy(wn, lang)
        XLIT_CACHE[ck] = translit
        out.append(translit)
    return " ".join(out)


In [ ]:
# ---- Normalization (full preprocessing.md §3; MIT-safe, no unidecode/GPL) ----
try:
    from anyascii import anyascii
    HAS_AA = True
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anyascii"])
    from anyascii import anyascii
    HAS_AA = True
print("anyascii available:", HAS_AA, "| e.g.", anyascii("Pr\u00edvate \u0926\u093f\u0932\u094d\u0932\u0940"))
JUNK_RE = re.compile(r"(\*+|>>|--|<<|##|\[services\]|\(id:\s*\d+\)|<null>)", re.I)
HON_RE = re.compile(r"^\s*(smt|shri|mr|mrs|miss|dr)\.?\s+", re.I)
SPACE_RE = re.compile(r"\s+")
DOMAIN_RE = re.compile(r"^([\w\-]+)\.(com|in|fr|co\.in|net|org)(\.\w+)?$", re.I)
HOUSE_RE = re.compile(r"(?:\b(?:h\.?\s*no\.?|house|plot|door)\s*\.?\s*#?\s*(\d+[a-z]?)|#\s*(\d+))", re.I)
PO_RE = re.compile(r"\b(p\.?o\.?\s*box|unit\s*\d+|hn\b|door\s*no)\b[^,]*,?", re.I)
PIN5_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")
PIN6_RE = re.compile(r"(?<!\d)(\d{6})(?!\d)")
LAND_RE = re.compile(r"\b(near|opp|opposite|behind|atm|temple|mosque|church|bank|hospital|station|mall)\b", re.I)
LEGAL = {"pvt", "private", "ltd", "limited", "corp", "corporation", "inc", "llc", "llp",
         "co", "enterprises", "sarl", "sas", "sa", "eurl", "sci", "groupe", "s.a.s"}
ABBR = {"rd": "road", "st": "street", "cir": "circle", "ave": "avenue", "av": "avenue",
        "blvd": "boulevard", "bd": "boulevard", "r": "rue"}
US_CODES = {"AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN",
            "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV",
            "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN",
            "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC"}
US_NAMES = {"alabama": "AL", "alaska": "AK", "arizona": "AZ", "california": "CA", "texas": "TX",
            "florida": "FL", "new york": "NY", "ohio": "OH", "oklahoma": "OK", "maryland": "MD",
            "iowa": "IA", "north carolina": "NC"}
IN_MAP = {"MH": "maharashtra", "RJ": "rajasthan", "TN": "tamil nadu", "KA": "karnataka",
          "DL": "delhi", "WB": "west bengal", "UP": "uttar pradesh", "GJ": "gujarat"}
IN_NAMES = {"maharashtra", "rajasthan", "tamil nadu", "karnataka", "delhi", "west bengal",
            "uttar pradesh", "gujarat", "kolkata", "howrah"}
NATIVE_STATE = {"\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bc1": "tamil nadu",
                "\u092e\u0939\u093e\u0930\u093e\u0937\u094d\u091f\u094d\u0930": "maharashtra",
                "\u0926\u093f\u0932\u094d\u0932\u0940": "delhi",
                "\u092a\u0936\u094d\u091a\u093f\u092e \u092c\u0902\u0917\u093e\u0932": "west bengal",
                "\u0915\u0930\u094d\u0928\u093e\u091f\u0915": "karnataka"}

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def soundex(s):
    s = re.sub(r"[^a-z]", "", s.lower())
    if not s:
        return ""
    mapping = {"b": "1", "f": "1", "p": "1", "v": "1", "c": "2", "g": "2", "j": "2",
               "k": "2", "q": "2", "s": "2", "x": "2", "z": "2", "d": "3", "t": "3",
               "l": "4", "m": "5", "n": "5", "r": "6"}
    out = s[0].upper()
    prev = mapping.get(s[0], "0")
    for ch in s[1:]:
        if ch in "hw":
            continue
        d = mapping.get(ch, "0")
        if d != "0" and d != prev:
            out += d
            if len(out) == 4:
                break
        prev = d
    return (out + "000")[:4]

def base_clean(s):
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = strip_accents(s).lower()
    s = JUNK_RE.sub(" ", s)
    return SPACE_RE.sub(" ", s).strip(" ,.-")

def normalize_name(raw):
    raw = "" if raw is None else str(raw)
    if XLIT_MODEL is not None and has_native_script(raw):
        raw = xlit_to_latin(raw)  # native words -> IndicXlit, Latin words passthrough
    s = base_clean("" if raw is None else str(raw))
    hon = int(bool(HON_RE.search(s)))
    s = HON_RE.sub("", s)
    m = DOMAIN_RE.match(s.replace(" ", ""))
    stem = m.group(1) if m else ""
    toks = [t.strip(".") for t in s.split(" ") if t]
    legal = [t for t in toks if t in LEGAL]
    core_toks = [t for t in toks if t not in LEGAL]
    if not core_toks:
        core_toks = toks  # name is only a legal word (e.g. "Private"); keep it as core
    core = " ".join(core_toks)
    return {"name_norm": s, "core_name": core,
            "legal_form": " ".join(t for t in toks if t in LEGAL), "domain_stem": stem,
            "name_lat": anyascii(core) if HAS_AA else core,
            "phonetic_code": soundex(core.split(" ")[0]) if core else "",
            "has_honorific": hon}

def extract_city_state(s):
    parts = [p.strip() for p in s.split(",") if p.strip()]
    city, state = "", ""
    if parts:
        last = parts[-1]
        toks = last.split()
        cand1 = toks[-1].upper().strip(".") if toks else ""
        cand2 = " ".join(toks[-2:]) if len(toks) >= 2 else ""
        if cand1 in US_CODES:
            state = cand1
        elif cand2 in US_NAMES:
            state = US_NAMES[cand2]
        elif cand2 in IN_NAMES or cand2 in IN_MAP.values():
            state = cand2
        elif toks and toks[-1].upper() in IN_MAP:
            state = IN_MAP[toks[-1].upper()]
        else:
            for tok in toks:
                if tok in NATIVE_STATE:
                    state = NATIVE_STATE[tok]
                    break
        if state and len(parts) >= 2:
            city = re.sub(r"\d+", "", parts[-2]).strip()
    return city, state

def normalize_address(raw):
    raw = "" if raw is None else str(raw)
    if XLIT_MODEL is not None and has_native_script(raw):
        raw = xlit_to_latin(raw)  # same hook as normalize_name
    s = base_clean("" if raw is None else str(raw))
    mh = HOUSE_RE.search(s)
    house = (mh.group(1) or mh.group(2) or "").lower() if mh else ""
    m6, m5 = PIN6_RE.search(s), PIN5_RE.search(s)
    pin = m6.group(1) if m6 else (m5.group(1) if m5 else "")
    land = int(bool(LAND_RE.search(s)))
    s2 = PO_RE.sub(" ", s)
    toks = [ABBR.get(t.strip("."), t.strip(".")) for t in s2.replace(",", " ").split()]
    norm = SPACE_RE.sub(" ", " ".join(toks)).strip()
    city, state = extract_city_state(s)
    return {"addr_norm": norm, "street_tokens": " ".join(sorted(set(norm.split()))),
            "house_no": house, "zip_pin_cp": pin, "city": city, "state_code": state,
            "empty_addr": int(norm == ""), "missing_pin": int(pin == ""),
            "missing_city": int(city == ""), "landmark_flag": land}

assert normalize_name("*** Pr\u00edvate *** [Services] (ID: 1072)")["core_name"] == "private"
assert normalize_name("GURU INFOTECH PVT. LTD.")["core_name"] == "guru infotech"
assert normalize_name("SARL Dupont")["legal_form"] == "sarl"
assert normalize_name("projectsmanipalindia.com")["domain_stem"] == "projectsmanipalindia"
assert normalize_name("Shri Ram Traders")["has_honorific"] == 1
assert normalize_address("H.no 780, R. Voltaire, 75011 Paris")["house_no"] == "780"
assert normalize_address("H.no 780, R. Voltaire, 75011 Paris")["zip_pin_cp"] == "75011"
assert normalize_address("175 Boulevard Bd Saint-Germain, Paris")["addr_norm"].find("boulevard") >= 0
assert normalize_address("#12, Lake Town, Kolkata")["house_no"] == "12"
assert normalize_address("Shop 5, Near SBI ATM, Pune")["landmark_flag"] == 1
assert normalize_address("")["empty_addr"] == 1
if HAS_AA:
    assert anyascii("\u0926\u093f\u0932\u094d\u0932\u0940").strip() != ""
print("all normalize unit tests PASS (anyascii=%s)" % HAS_AA)


In [ ]:
# ---- Run: per-country parquet (preprocessing.md §4 contract) ----
NEW_COLS = ["name_norm", "core_name", "legal_form", "domain_stem", "name_lat",
            "phonetic_code", "has_honorific", "addr_norm", "street_tokens", "house_no",
            "zip_pin_cp", "city", "state_code", "empty_addr", "missing_pin",
            "missing_city", "landmark_flag"]

def normalize_frame(df):
    names = [normalize_name(x) for x in df["business_name"].to_list()]
    addrs = [normalize_address(x) for x in df["business_address"].to_list()]
    cols = [[n["name_norm"] for n in names], [n["core_name"] for n in names],
            [n["legal_form"] for n in names], [n["domain_stem"] for n in names],
            [n["name_lat"] for n in names], [n["phonetic_code"] for n in names],
            [n["has_honorific"] for n in names], [a["addr_norm"] for a in addrs],
            [a["street_tokens"] for a in addrs], [a["house_no"] for a in addrs],
            [a["zip_pin_cp"] for a in addrs], [a["city"] for a in addrs],
            [a["state_code"] for a in addrs], [a["empty_addr"] for a in addrs],
            [a["missing_pin"] for a in addrs], [a["missing_city"] for a in addrs],
            [a["landmark_flag"] for a in addrs]]
    return df.with_columns([pl.Series(c, v) for c, v in zip(NEW_COLS, cols)])

def countries_of(src):
    return [r["country"] for r in pl.scan_csv(str(src), separator="\t",
            infer_schema_length=10000).select("country").unique().collect().to_dicts()]

SAMPLED_S1 = {}          # country -> entity_ids of the sampled train S1 (linked mode)
S1_ROWS = {}             # (split, country) -> S1 rows in the full file, for the S2/S3 ratio
MC_BY_S1 = pairs.group_by("s1").len().rename({"s1": "entity_id", "len": "_mc"})   # train S1 -> match count (diversity)


def process_sample_per_country(src, stem, quota_each=None):
    """A sample of every country of one file. S1: SAMPLE_N_PER_COUNTRY rows; S2/S3: the full data's ratio to S1.
    Rows are chosen by SAMPLE_MODE (linked: true matches of the sampled train S1 first) and, unless
    SAMPLE_DIVERSE is off, stratified across name shape (native-script / domain / honorific) and, for train
    S1 itself, match count -- see add_case_tag / add_mc_tag / stratified_sample in the settings cell."""
    split, source = stem.split("_")
    out = {}
    for c in countries_of(src):
        lf = pl.scan_csv(str(src), separator="	", infer_schema_length=10000).filter(pl.col("country") == c)
        total = lf.select(pl.len()).collect().item()
        if source == "s1":
            S1_ROWS[(split, c)] = total
            n = quota_each or SAMPLE_N_PER_COUNTRY
        else:
            n = quota_each or round(SAMPLE_N_PER_COUNTRY * total / S1_ROWS.get((split, c), total))
        n = min(n, total)
        if SAMPLE_MODE == "head":
            df = lf.head(n).collect()
        elif SAMPLE_MODE == "linked" and split == "train" and source != "s1" and c in SAMPLED_S1:
            sampled = pl.DataFrame({"s1": SAMPLED_S1[c]})
            wanted = pl.DataFrame({"entity_id": pairs.join(sampled, on="s1", how="semi")["mid"]}).lazy()
            true_rows = lf.join(wanted, on="entity_id", how="semi").collect()      # true matches of the sampled S1
            if len(true_rows) > n:
                true_rows = stratified_sample(add_case_tag(true_rows), n, ["_case"], SEED)
            rest = lf.join(wanted, on="entity_id", how="anti").collect()
            fill = stratified_sample(add_case_tag(rest), min(n - len(true_rows), len(rest)), ["_case"], SEED)
            df = pl.concat([true_rows, fill]).sample(fraction=1.0, shuffle=True, seed=SEED)
            print(f"  {stem} {c}: {len(true_rows):,} true matches of the sampled S1 + {len(fill):,} other records")
        else:                                                                  # random (and linked S1 / test files)
            tagged = add_case_tag(lf.collect())
            tag_cols = ["_case"]
            if split == "train" and source == "s1":                           # only S1 has a known match count
                tagged = add_mc_tag(tagged, MC_BY_S1)
                tag_cols.append("_mc")
            df = stratified_sample(tagged, n, tag_cols, SEED)
            if SAMPLE_MODE == "linked" and stem == "train_s1":
                SAMPLED_S1[c] = df["entity_id"]
        df = normalize_frame(df)
        dst = OUT / f"{stem}_{c}.parquet"
        df.write_parquet(str(dst))
        out[c] = len(df)
        print(f"WROTE {dst.name} rows={len(df)} of {total:,} (sample, {SAMPLE_MODE})")
    return out


def process_full_per_country(src, stem, chunk=CHUNK):
    total = pl.scan_csv(str(src), separator="\t", infer_schema_length=10000).select(pl.len()).collect().item()
    tmpdir = OUT / f"_tmp_{stem}"
    tmpdir.mkdir(exist_ok=True)
    seen = set()
    for off in range(0, total, chunk):
        df = pl.scan_csv(str(src), separator="\t", infer_schema_length=10000).slice(off, chunk).collect()
        df = normalize_frame(df)
        for c in df["country"].unique().to_list():
            part = df.filter(pl.col("country") == c)
            p = tmpdir / f"{c}_{off}.parquet"
            part.write_parquet(str(p))
            seen.add(c)
        print(f"{Path(src).name} {off + len(df)}/{total}")
    out = {}
    for c in sorted(seen):
        full = pl.concat([pl.read_parquet(str(f))
                          for f in sorted(tmpdir.glob(f"{c}_*.parquet"))], how="diagonal")
        dst = OUT / f"{stem}_{c}.parquet"
        full.write_parquet(str(dst))
        out[c] = len(full)
        print(f"WROTE {dst.name} rows={len(full)}")
    for f in tmpdir.glob("*.parquet"):
        f.unlink()
    tmpdir.rmdir()
    return out

counts = {}
SPEC = [("s1tr", "train_s1"), ("s1te", "test_s1"), ("s2tr", "train_s2"), ("s3tr", "train_s3"),
        ("s2te", "test_s2"), ("s3te", "test_s3")]   # test S2/S3 are needed for the submission
for k, stem in SPEC:
    if stem.split("_")[0] not in PROCESS_SPLITS:
        continue
    if FULL_RUN:
        counts[k] = process_full_per_country(P[k], stem)
    else:
        counts[k] = process_sample_per_country(P[k], stem)
counts["_settings"] = {"full_run": FULL_RUN} if FULL_RUN else {"full_run": False, "sample_s1_per_country": SAMPLE_N_PER_COUNTRY,
                                                             "sample_mode": SAMPLE_MODE, "seed": SEED}
json.dump(counts, open(OUT / "processed_counts.json", "w"), indent=1)
print("DONE:", counts)


In [ ]:
# ---- Verify: before/after, FR inspection, schema freeze ----
stem = "train_s1"
pat = str(OUT / f"{stem}_US.parquet")
df = pl.read_parquet(pat).select(["business_name", "core_name", "legal_form", "name_lat",
    "business_address", "addr_norm", "house_no", "zip_pin_cp", "city", "state_code"]).head(5)
for r in df.to_dicts():
    print(safe(r["business_name"]), "->", safe(r["core_name"]), "| legal=", safe(r["legal_form"]),
          "| lat=", safe(r["name_lat"]))
    print("   ", safe(r["business_address"]), "->", safe(r["addr_norm"]),
          "| house=", r["house_no"], "pin=", r["zip_pin_cp"], "city=", safe(r["city"]), "st=", safe(r["state_code"]))
frf = OUT / "test_s1_France.parquet"
if frf.exists():                                   # absent when only train was processed
    fr = pl.read_parquet(str(frf)).select(["business_name", "core_name", "business_address", "addr_norm"]).head(3)
    print("--- FR inspection (test S1 France) ---")
    for r in fr.to_dicts():
        print(safe(r["business_name"]), "->", safe(r["core_name"]))
        print("   ", safe(r["business_address"]), "->", safe(r["addr_norm"]))
print("schema=", pl.read_parquet(pat).columns)
print("counts=", json.load(open(OUT / "processed_counts.json")))
print("verify OK")
